In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/datatran2017.csv", sep=";", encoding="latin1")

df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,17,2017-01-01,domingo,01:45:00,RS,116,"34,9",VACARIA,Defeito Mecânico no Veículo,Colisão traseira,...,0,2,0,4,2,"-28,5071196","-50,941176",SPRF-RS,DEL05-RS,UOP03-DEL05-RS
1,20,2017-01-01,domingo,01:00:00,PR,376,636,TIJUCAS DO SUL,Velocidade Incompatível,Saída de leito carroçável,...,0,1,0,0,2,"-25,754","-49,1266",SPRF-PR,DEL01-PR,DEL7/1-UOP08/PR
2,69,2017-01-01,domingo,04:40:00,BA,101,65,ENTRE RIOS,Condutor Dormindo,Colisão frontal,...,1,2,0,2,2,"-11,9618","-38,0953",SPRF-BA,DEL01-BA,UOP04-DEL01-BA
3,106,2017-01-01,domingo,06:30:00,PA,316,"72,5",CASTANHAL,Falta de Atenção à Condução,Colisão lateral,...,0,3,0,0,3,"-1,2899799","-47,83483207",SPRF-PA,DEL01-PA,DEL19/1-UOP02/PA
4,109,2017-01-01,domingo,09:00:00,GO,20,"220,5",POSSE,Defeito na Via,Colisão com objeto estático,...,1,0,1,3,2,"-14,14220931","-46,32258922",SPRF-DF,DEL02-DF,UOP03-DEL02-DF


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 89567 entries, 0 to 89566
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   id                      89567 non-null  int64
 1   data_inversa            89567 non-null  str  
 2   dia_semana              89567 non-null  str  
 3   horario                 89567 non-null  str  
 4   uf                      89567 non-null  str  
 5   br                      89567 non-null  int64
 6   km                      89567 non-null  str  
 7   municipio               89567 non-null  str  
 8   causa_acidente          89567 non-null  str  
 9   tipo_acidente           89567 non-null  str  
 10  classificacao_acidente  89566 non-null  str  
 11  fase_dia                89567 non-null  str  
 12  sentido_via             89567 non-null  str  
 13  condicao_metereologica  89567 non-null  str  
 14  tipo_pista              89567 non-null  str  
 15  tracado_via             89567 

In [3]:
for coluna in df.select_dtypes(include="object").columns:
    if df[coluna].nunique() < 50:
        print(f"\n=== {coluna} ===")
        print(df[coluna].value_counts(dropna=False))


=== dia_semana ===
dia_semana
sábado           15175
domingo          14785
sexta-feira      13849
segunda-feira    12175
quinta-feira     11788
quarta-feira     11030
terça-feira      10765
Name: count, dtype: int64

=== uf ===
uf
MG    12730
PR    10689
SC    10665
RS     6386
SP     6011
RJ     5972
BA     4687
GO     4370
MT     3526
PE     3441
ES     3004
CE     1954
MS     1933
PB     1852
RO     1831
MA     1583
PI     1540
PA     1504
RN     1457
DF     1093
AL      904
TO      785
SE      763
AC      304
RR      286
AP      166
AM      131
Name: count, dtype: int64

=== causa_acidente ===
causa_acidente
Falta de Atenção à Condução                                                       34510
Velocidade Incompatível                                                           10432
Ingestão de Álcool                                                                 6448
Não guardar distância de segurança                                                 5837
Desobediência às normas de

/tmp/ipykernel_21457/538391312.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for coluna in df.select_dtypes(include="object").columns:


In [4]:
df["fatal"] = df["classificacao_acidente"] == "Com Vítimas Fatais"
frontal = df[df["tipo_acidente"] == "Colisão frontal"]

tabela = pd.DataFrame({
    "Acidentes": df.groupby("tipo_pista").size(),
    "% fatais": df.groupby("tipo_pista")["fatal"].mean() * 100,
    "% fatais em colisão frontal": frontal.groupby("tipo_pista")["fatal"].mean() * 100,
}).round(1)

tabela.loc[["Simples", "Dupla", "Múltipla"]]


,Acidentes,% fatais,% fatais em colisão frontal
tipo_pista,,,
Simples,46347,7.7,29.7
Dupla,37243,3.7,15.1
Múltipla,5977,3.5,12.9


Pergunta central: o que faz um acidente numa rodovia federal virar um acidente com morte?

Hipótese de intervenção:

- Duplicar trechos de pista simples reduz a chance de um acidente ser fatal.
- Se a hipótese se sustentar, recomendamos priorizar a duplicação dos trechos onde o modelo prevê o maior ganho.